# The Correlation Coefficient

Age–height has covariance $17$; age–naps has $-7.45$. Is the first relationship *stronger*? You cannot tell — covariance has no fixed range, and its size depends on the variables' units. The fix is the **correlation coefficient**: divide the covariance by both standard deviations. The result always lies between $-1$ and $1$ — a universal, unit-free strength-of-relationship scale: $-1$ for a perfect falling line, $+1$ for a perfect rising line, $0$ for no linear relation at all. It is based on the ninth video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

The covariance's *sign* is meaningful, but its *magnitude* is not comparable across datasets: 17 might just mean "big numbers" rather than "strong relationship". This is the same units problem the standard deviation solved for the variance — and the solution is again standardization.

After completing this notebook you will be able to:

- Compute the correlation coefficient $\rho = \frac{\mathrm{Cov}(X, Y)}{\sigma_X \sigma_Y}$ and place it on the $[-1, 1]$ scale.
- Interpret $\rho \approx \pm 1$ (near-perfect lines), $\rho \approx 0$ (no linear relation), and everything between.
- Show that $\rho$ is immune to units and shifts — and equal to the covariance of the *standardized* variables.
- Know the classic caution: correlation summarizes *linear* association only, and identical statistics can hide very different data.

## 2. Theory

### Covariance magnitudes are not comparable

Age vs naps: covariance $-7.45$; age vs height: $+17$. Forgetting the signs, is the height relationship stronger because $17 > 7.45$? We can't tell — it could be that the heights are just much bigger numbers. The covariance can take any value, with no limited range.

### The correlation coefficient

The **correlation coefficient** is a number that always lies between $-1$ and $1$:

- $\rho = -1$: completely negatively correlated (the data sits on a falling line, like $y = -x$);
- $\rho = +1$: completely positively correlated (a rising line, like $y = x$);
- $\rho = 0$: no linear relationship (e.g. independent variables).

The formula is simply the covariance, **standardized** — divided by both standard deviations:

$$
\rho_{XY} = \frac{\mathrm{Cov}(X, Y)}{\sigma_X\,\sigma_Y} = \frac{\mathrm{Cov}(X, Y)}{\sqrt{\mathrm{Var}(X)}\sqrt{\mathrm{Var}(Y)}}
$$

Equivalently: $\rho$ is the covariance of the standardized variables $\frac{X-\mu_X}{\sigma_X}$ and $\frac{Y-\mu_Y}{\sigma_Y}$ — center and scale first, then covary. That is why it is unit-free and bounded (the Cauchy–Schwarz bound $|\mathrm{Cov}| \le \sigma_X\sigma_Y$ from the last notebook is exactly the statement $|\rho| \le 1$).

### The video's computations

- **Age vs naps:** $\rho = \frac{-7.45}{\sqrt{9.17}\sqrt{7.57}} = -0.894$ — close to $-1$: pretty much completely negatively correlated.
- **Age vs height:** $\rho = \frac{17}{\sqrt{9.17}\sqrt{39.56}} = 0.893$ — strongly positively correlated.
- **Age vs grades:** $\rho = \frac{0.1}{\sqrt{9.17}\sqrt{9.78}} \approx 0.01$ — definitely uncorrelated.
- **Call center (wait vs rating):** $\rho = \frac{-7.878}{\sqrt{8.526}\sqrt{10.163}} = -0.845$ — strongly negative, as the falling diagonal suggested.

Despite the wildly different covariance magnitudes ($17$ vs $-7.45$ vs $0.1$ vs $-7.878$), every correlation lands neatly in $[-1, 1]$, and now the comparisons are meaningful.

> **Convention note (small erratum):** the video divides an $n$-based covariance ($17 = 170/10$) by $(n{-}1)$-based standard deviations ($\sqrt{9.17}$), giving $0.893$. Using one convention consistently (either both $\div n$ or both $\div (n{-}1)$ — the choice cancels in the ratio!), the age–height correlation is $0.992$ and age–naps is $-0.994$ — even closer to perfect lines, matching how straight the scatters look. The lesson stands either way; just compute numerator and denominator with the same convention (as `np.corrcoef` does).

## 3. Worked Examples

### Example 1: consistent-convention correlations for the children

With population ($\div n$) values everywhere — $\mathrm{Var}(\text{age}) = 8.25$, $\mathrm{Var}(\text{height}) = 35.6$, $\mathrm{Var}(\text{naps}) = 6.81$, $\mathrm{Var}(\text{grades}) = 8.8$:

$$
\rho_{\text{age,height}} = \frac{17}{\sqrt{8.25 \cdot 35.6}} = 0.992
\qquad
\rho_{\text{age,naps}} = \frac{-7.45}{\sqrt{8.25 \cdot 6.81}} = -0.994
\qquad
\rho_{\text{age,grades}} = \frac{0.1}{\sqrt{8.25 \cdot 8.8}} = 0.012
$$

### Example 2: the games

Games 1–3 had $\sigma_X = \sigma_Y = 1$, so their correlations equal their covariances: exactly $+1$, $-1$, and $0$ — the three landmark values realized by real games (win together / zero-sum / independent).

### Example 3: units cannot move it

Convert heights to centimeters: the covariance jumps from $17$ to $43.18$, but $\sigma_{\text{height}}$ grows by the same factor $2.54$, so $\rho$ is untouched. Correlation compares *relationships*, not *scales*.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

age = np.array([6, 7, 8, 9, 10, 11, 12, 13, 14, 15])
height = np.array([50, 52, 55, 57, 60, 62, 64, 65, 67, 68])
grades = np.array([5, 7, 8, 3, 1, 1, 6, 10, 2, 7])
naps = np.array([8, 7, 6, 5, 4, 3, 2, 1, 1, 0])

def corr(x, y):
    cov = ((x - x.mean()) * (y - y.mean())).mean()
    return cov / (x.std() * y.std())

### The three children correlations — and the convention check

In [ ]:
for y, name in [(height, "height"), (grades, "grades"), (naps, "naps")]:
    print(f"rho(age, {name:<6}) = {corr(age, y):>7.3f}"
          f"   (np.corrcoef: {np.corrcoef(age, y)[0, 1]:>7.3f})")

print()
video_style = 17 / (np.sqrt(9.17) * np.sqrt(39.56))
print(f"the video's mixed-convention 17/sqrt(9.17 * 39.56) = {video_style:.3f}")
print("Same convention top and bottom (as above) gives 0.992 -- and the ddof")
print("choice cancels: n-based and (n-1)-based ratios agree exactly.")

### Correlation = covariance of standardized variables

In [ ]:
zx = (age - age.mean()) / age.std()
zh = (height - height.mean()) / height.std()
print(f"Cov(z_age, z_height) = {(zx * zh).mean():.4f}")
print(f"rho(age, height)     = {corr(age, height):.4f}")
print("Standardize first, then covary -- the two roads meet.")

### Immune to units and shifts (unlike covariance)

In [ ]:
height_cm = height * 2.54
height_shift = height + 100

print(f"Cov changes with units:  inches {((age - 10.5) * (height - 60)).mean():.2f}"
      f" -> cm {((age - 10.5) * (height_cm - height_cm.mean())).mean():.2f}")
print(f"rho does not:            inches {corr(age, height):.4f}"
      f" -> cm {corr(age, height_cm):.4f} -> shifted {corr(age, height_shift):.4f}")
print(f"flip the sign of one variable: rho(age, -height) = {corr(age, -height):.4f}")

### The games and the call center

In [ ]:
rng = np.random.default_rng(42)

# games: sample many rounds and correlate the players' outcomes
def play(outcomes, n=200_000):
    p = np.array([o[0] for o in outcomes])
    idx = rng.choice(len(outcomes), p=p, size=n)
    return (np.array([outcomes[i][1] for i in idx], dtype=float),
            np.array([outcomes[i][2] for i in idx], dtype=float))

for name, outcomes in [("game 1", [(1/2, 1, 1), (1/2, -1, -1)]),
                       ("game 2", [(1/2, 1, -1), (1/2, -1, 1)]),
                       ("game 3", [(1/4, 1, 1), (1/4, 1, -1), (1/4, -1, 1), (1/4, -1, -1)])]:
    x, y = play(outcomes)
    print(f"{name}: rho = {corr(x, y):>7.3f}")

# call center
n = 400_000
quick = rng.random(n) < 0.5
wait = np.clip(np.where(quick, rng.normal(2.4, 1.6, n), rng.normal(7.6, 1.6, n)), 0, 10)
rate = np.clip(np.where(quick, rng.normal(8.0, 1.7, n), rng.normal(2.6, 1.7, n)), 0, 10)
print(f"call center: rho = {corr(wait, rate):.3f}   (video: -0.845)")

## 5. Visualizations

### The three children scatters, now with comparable numbers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, y, name in [(axes[0], height, "height"), (axes[1], grades, "grades"),
                    (axes[2], naps, "naps per day")]:
    r = corr(age, y)
    ax.scatter(age, y, s=60, color="tab:blue", zorder=3)
    ax.set_title(f"age vs {name}\n$\\rho$ = {r:.3f}")
    ax.set_xlabel("age")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
plt.suptitle("One unit-free scale for all three relationships")
plt.tight_layout()
plt.show()

### A correlation gallery

Synthetic clouds spanning the whole scale — train your eye for what each $\rho$ looks like.

In [ ]:
rng = np.random.default_rng(1)

fig, axes = plt.subplots(1, 7, figsize=(16, 2.8), sharex=True, sharey=True)
targets = [-1, -0.9, -0.5, 0, 0.5, 0.9, 1]
n = 300
for ax, t in zip(axes, targets):
    x = rng.standard_normal(n)
    if abs(t) == 1:
        y = t * x
    else:
        y = t * x + np.sqrt(1 - t**2) * rng.standard_normal(n)
    ax.scatter(x, y, s=6, alpha=0.6, color="tab:blue")
    ax.set_title(f"$\\rho$ = {corr(x, y):.2f}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("From perfect negative through none to perfect positive")
plt.tight_layout()
plt.show()

### Same correlation, different stories: Anscombe's quartet

The famous warning (from the lab the video announces): four datasets with nearly identical means, variances, and correlation $\rho \approx 0.816$ — and utterly different shapes. Summary statistics never replace looking at the data.

In [ ]:
anscombe_x = [np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5])] * 3 + \
             [np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8])]
anscombe_y = [
    np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharey=True)
for i, (ax, x, y) in enumerate(zip(axes, anscombe_x, anscombe_y), 1):
    ax.scatter(x, y, s=40, color="tab:blue", zorder=3)
    ax.set_title(f"dataset {i}: $\\rho$ = {corr(x, y):.3f}", fontsize=11)
    ax.set_xlabel("x")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("y")
plt.suptitle("Anscombe's quartet: identical correlations, very different data")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** A dataset has $\mathrm{Cov}(X,Y) = 30$, $\sigma_X = 5$, $\sigma_Y = 8$. Compute $\rho$. Another has $\mathrm{Cov} = 0.9$, $\sigma_X = 1$, $\sigma_Y = 1$. Which relationship is stronger?

**Exercise 2.** What is $\rho(X, X)$? And $\rho(X, -3X + 7)$? Verify both on the age data and explain via the formula.

### Intermediate

**Exercise 3.** For game 4 ($(1,1)$ w.p. $\tfrac12$, $(-1,-1)$ w.p. $\tfrac13$, $(0,0)$ w.p. $\tfrac16$; $\mathrm{Var} = \mathrm{Cov} = \tfrac{29}{36}$), compute $\rho$ *from the distribution*. Why was the answer predictable without any arithmetic?

**Exercise 4.** Show numerically that adding independent noise dilutes correlation: for $Y = X + \epsilon$ with $X \sim N(0,1)$ and noise $\epsilon \sim N(0, s^2)$, compute $\rho$ for $s = 0, 0.5, 1, 2, 5$ and compare with the theory $\rho = \frac{1}{\sqrt{1 + s^2}}$.

### Advanced

**Exercise 5.** Correlation vs causation vs nonlinearity, all in one: (a) the parabola $y = x^2$ on $x \in \{-2..2\}$ has $\rho = 0$ — verify; (b) shoe size and reading ability in children are strongly correlated — construct such data where both are driven by age, and show the correlation vanishes *within* a fixed age group. What do (a) and (b) each warn against?

**Exercise 6.** The **correlation matrix**: compute it for the four children variables (divide each covariance-matrix cell by $\sigma_i \sigma_j$, or standardize the data and take `np.cov`), confirm the diagonal is all 1's, and visualize as a heatmap. Why is this the version usually shown in data-science reports rather than the raw covariance matrix?

## 7. Solutions

In [ ]:
# Exercise 1: comparable at last
rho1 = 30 / (5 * 8)
rho2 = 0.9 / (1 * 1)
print(f"Exercise 1: rho_1 = {rho1}, rho_2 = {rho2}")
print("The SECOND is stronger (0.9 > 0.75) despite its covariance being 33x")
print("smaller -- exactly the comparison covariance alone could not make.")

In [ ]:
# Exercise 2: self-correlation and perfect linear maps
print(f"Exercise 2: rho(age, age)       = {corr(age, age):.1f}")
print(f"            rho(age, -3*age+7)  = {corr(age, -3 * age + 7):.1f}")
print("rho(X, X) = Var/ (sigma*sigma) = 1. Any exact linear map aX + b gives")
print("|rho| = 1 with the sign of a: correlation measures LINE-ness, and a")
print("perfect line is as line-y as it gets.")

In [ ]:
# Exercise 3: game 4's correlation
rho_g4 = (29 / 36) / (np.sqrt(29 / 36) * np.sqrt(29 / 36))
print(f"Exercise 3: rho = (29/36) / (29/36) = {rho_g4:.1f}")
print("Predictable: in game 4, Y = X in every outcome -- a perfect rising")
print("'line' -- so the correlation had to be exactly +1.")

In [ ]:
# Exercise 4: noise dilutes correlation
rng = np.random.default_rng(0)
x = rng.standard_normal(500_000)
print("Exercise 4:")
print(f"{'s':>5} {'rho (sim)':>10} {'theory':>8}")
for s in [0, 0.5, 1, 2, 5]:
    y = x + s * rng.standard_normal(500_000)
    print(f"{s:>5} {corr(x, y):>10.3f} {1/np.sqrt(1 + s**2):>8.3f}")

**Exercise 4, illustrated.** A one-page visual summary of the solution above.

<img src="img/09_ex4_noise_dilutes_correlation.png" alt="Illustrated solution to Exercise 4" width="900">

In [ ]:
# Exercise 5: two classic cautions
x_par = np.array([-2, -1, 0, 1, 2], dtype=float)
print(f"Exercise 5a: rho(x, x^2) = {corr(x_par, x_par**2):.3f} -- zero despite")
print("perfect (nonlinear) dependence: rho only sees LINES.\n")

rng = np.random.default_rng(0)
n = 30_000
kid_age = rng.uniform(6, 12, n)
shoe = 0.9 * kid_age + rng.normal(0, 0.6, n)
reading = 10 * kid_age + rng.normal(0, 8, n)
in_group = (kid_age > 8.9) & (kid_age < 9.1)
print(f"Exercise 5b: rho(shoe, reading) overall       = {corr(shoe, reading):.3f}")
print(f"             rho(shoe, reading) among 9-year-olds = "
      f"{corr(shoe[in_group], reading[in_group]):.3f}")
print("(a) warns: rho = 0 does not mean 'unrelated'.")
print("(b) warns: rho >> 0 does not mean 'causal' -- age drives both, and the")
print("correlation dissolves once the common cause is held fixed.")

**Exercise 5, illustrated.** A one-page visual summary of the solution above.

<img src="img/09_ex5_correlation_cautions.png" alt="Illustrated solution to Exercise 5" width="900">

In [ ]:
# Exercise 6: the correlation matrix
variables = {"age": age, "height": height, "grades": grades, "naps": naps}
names = list(variables)
R = pd.DataFrame([[corr(variables[a], variables[b]) for b in names]
                  for a in names], index=names, columns=names)
print("Exercise 6:")
print(R.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(R.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(names)
ax.set_yticks(range(4)); ax.set_yticklabels(names)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{R.values[i, j]:.2f}", ha="center", va="center")
ax.set_title("correlation matrix: unit-free, diagonal = 1, fixed color scale")
fig.colorbar(im, label=r"$\rho$")
plt.tight_layout()
plt.show()

print("Every entry is comparable on the same [-1, 1] scale with a universal")
print("color map -- raw covariances would need a different scale per pair of")
print("units, which is why reports almost always show correlations.")

## 8. Conclusion

- Covariance magnitudes are incomparable across datasets ($17$ vs $-7.45$ says nothing about relative strength); the **correlation coefficient** fixes this by standardizing: $\rho = \frac{\mathrm{Cov}(X,Y)}{\sigma_X \sigma_Y}$, always in $[-1, 1]$.
- Landmarks: $-1$ = perfect falling line ($y = -x$), $+1$ = perfect rising line ($y = x$), $0$ = no linear relationship. The games realize all three exactly.
- The lesson's datasets, now comparable: age–height $\approx +0.99$, age–naps $\approx -0.99$, age–grades $\approx 0.01$, call center $-0.845$. (The video's $\pm 0.89$ came from mixing $\div n$ and $\div(n{-}1)$ conventions — use one consistently and the ddof choice cancels entirely.)
- $\rho$ is the covariance of standardized variables: immune to units, shifts, and positive scalings; a sign flip in one variable flips $\rho$'s sign.
- Cautions, previewing the lab: $\rho$ measures **linear** association only (a parabola scores 0), correlation is not causation (shoe size vs reading), and identical statistics can hide different data (Anscombe's quartet) — always plot.

This completes the toolkit for pairs of variables: joint, marginal, and conditional distributions, covariance, the covariance matrix, and correlation.